# 迁移学习与分阶段解冻

## 学习目标

能够加载预训练 ResNet、冻结主干、替换分类头，并使用分组学习率解冻最后一层。


## 概念模型与执行路径

预训练模型提供通用视觉特征。第一阶段只训练新分类头；第二阶段以更小学习率微调靠近输出的主干层，避免快速破坏已有表示。输入必须使用预训练权重对应的尺寸和归一化。


### 实验 1：定位迁移学习入口

**实验目的**：定位课程根目录，使 notebook 能导入 `examples.transfer_learning`。路径搜索和 `sys.path` 修改只解决交互环境的启动目录差异；找不到 `common` 时应先检查工作目录。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：冻结 ResNet18 主干并替换分类头

**实验目的**：离线检查迁移学习模型的参数注册和冻结边界。`pretrained=False` 不下载权重，只创建 ResNet18 结构，因此本实验不能证明预训练效果。真实训练默认使用 ImageNet 预训练权重，quick 模式则同样跳过权重下载。

`build_model` 先把原模型所有参数设为 `requires_grad=False`，再用新的 `Linear(512,10)` 替换 `fc`。新层在冻结操作之后创建，默认可训练，所以解冻前只有分类头的 $512\times10+10=5130$ 个参数参与优化。

**观察重点**：冻结不会删除参数，也不会降低前向计算量；它避免保存这些参数的梯度并阻止 optimizer 更新。

In [ ]:
from examples.transfer_learning import build_model, unfreeze_last_block
model = build_model(pretrained=False)  # 离线检查结构；真实课程默认下载预训练权重
trainable_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable before unfreeze:", trainable_before)


### 实验 3：解冻最后一个残差阶段

**实验目的**：调用 `unfreeze_last_block` 将 `layer4` 的参数设为可训练，验证可训练参数量显著增加。layer4 靠近输出端，包含更任务相关的高级特征，通常比一次性解冻整个主干更稳妥。

断言只验证数量关系。由于本实验没有加载预训练权重，解冻行为仍是结构演示；迁移收益必须在使用预训练权重和正确输入预处理的真实数据上评估。

**BatchNorm 边界**：`requires_grad=False` 不会自动冻结运行均值和方差。只要整个模型处于 train 模式，冻结主干中的 BatchNorm buffer 仍可能更新；小数据微调时应明确决定是否让这些统计量变化。

In [ ]:
unfreeze_last_block(model)
trainable_after = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable after unfreeze:", trainable_after)
assert trainable_after > trainable_before


### 实验 4：为主干和分类头设置分组学习率

**实验目的**：创建两个 optimizer 参数组：已解冻的 `layer4` 使用 `1e-4`，随机初始化的 `fc` 使用 `1e-3`。打印结果应为 `[0.0001, 0.001]`。

分类头必须从头适应 CIFAR-10，允许较大更新；预训练层已有有用表示，用较小学习率可降低灾难性遗忘风险。参数组必须互不重复且只包含想更新的参数。若在 optimizer 创建后才解冻新参数，旧 optimizer 不会自动纳入它们，必须新增参数组或重建 optimizer。

**状态提醒**：重建 optimizer 会丢失已有动量/Adam 统计。课程脚本在阶段切换时重建 AdamW，这是清晰的教学实现，也意味着第二阶段 optimizer state 从头开始。

In [ ]:
import torch
optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])
print([group["lr"] for group in optimizer.param_groups])


### 实验 5：运行真实 CIFAR-10 分阶段微调

**实验目的**：通过命令行运行完整数据加载、冻结训练、阶段解冻、验证选择和测试。该单元只提供命令，不会在 notebook 内执行。

真实模式下载 ImageNet 预训练 ResNet18；输入被 resize 到 `224×224`，并使用对应权重的 ImageNet mean/std。训练集有随机水平翻转，验证和测试没有随机增强。脚本在 `max(2, epochs//2)` 解冻 layer4，分类头学习率为 `1e-3`、layer4 为 `1e-4`。

验证准确率改善时保存 checkpoint，metadata 记录是否已经解冻；测试前恢复最佳权重。`--quick` 使用随机初始化 ResNet，主要用于离线流程冒烟，不应把其结果称作迁移学习基线。恢复训练时脚本先读取 metadata 决定冻结状态，再构造匹配的 optimizer 并恢复其状态。

In [ ]:
# 真实 CIFAR-10 训练：
# python 07-deep-learning/pytorch/examples/transfer_learning.py --epochs 4 --batch-size 32
# 离线结构冒烟需要已有 CIFAR 缓存；--quick 不下载预训练权重。


## 底层机制

迁移学习依赖三个契约同时成立：模型确实加载预训练权重；输入尺寸和归一化与权重预期一致；优化策略不会过快破坏已有表示。冻结通过 `requires_grad=False` 控制参数梯度，但模型仍执行完整前向，BatchNorm buffer 也独立于参数冻结。

分阶段解冻是一种稳定性策略，不保证总是更好。域差异很大时需要解冻更多层；数据很少时可能只训练分类头更稳。应同时比较验证指标、训练时间、显存和可训练参数量。

## 检查点

回答并验证：1）解冻前为何恰有 5130 个可训练参数？2）冻结是否减少前向计算？3）分类头学习率为什么更高？4）optimizer 创建后再解冻会自动更新新参数吗？5）冻结参数为何不等于冻结 BatchNorm 统计？6）`--quick` 为什么不能证明迁移学习有效？7）错误归一化会破坏什么契约？

## 试一试

使用同一预训练权重、划分和种子，比较只训练分类头、解冻 layer4、全部解冻三种策略，记录验证准确率、时间、峰值显存和可训练参数量。再比较正确 ImageNet 归一化与错误 CIFAR 归一化，并尝试冻结 BatchNorm 统计，观察小数据场景的变化。

## 常见错误与调试

- **预训练模型使用错误归一化/尺寸**：输入分布偏离预训练契约；使用权重对应 transform。
- **一次性大幅更新所有层**：可能灾难性遗忘；分阶段解冻并降低主干学习率。
- **替换头后仍冻结或漏进 optimizer**：检查 `requires_grad` 和参数组。
- **解冻后沿用旧 optimizer**：新参数不会更新；重建或添加参数组。
- **忽略 BatchNorm buffer**：冻结参数仍可能改变统计；明确模式策略。
- **用随机权重/随机输入声称迁移成功**：只能算结构冒烟；真实结论需预训练权重和验证集。
- **恢复时冻结状态不匹配**：optimizer state 无法对应；从 metadata 重建阶段状态。